# Hands-On Data Lab Implementation
**Yuva Intern | Data Science Practical**

This notebook covers environment setup, data manipulation, data cleaning, visualization, and basic statistics using a fictional student performance dataset.

## Learning objectives
- Use Pandas and NumPy for loading, filtering, grouping, and numerical calculations.
- Detect and handle missing values and duplicate records.
- Create bar, line, histogram, scatter, and box plots.
- Summarize distributions and correlations.


## 1. Import libraries and set up environment

If a package is missing, run `python -m pip install -r requirements.txt` in the project terminal.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "student_performance.csv"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
print("Environment ready")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


## 2. Load and inspect the dataset

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
display(df_raw.head())
print("Dataset shape:", df_raw.shape)
print("\nData types:")
display(df_raw.dtypes.to_frame("data_type"))
print("\nMissing values:")
display(df_raw.isna().sum().to_frame("missing_count"))


## 3. Data cleaning

In [ ]:
df = df_raw.copy()
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["Student_ID"], keep="first")

numeric_cols = ["Study_Hours", "Attendance_Percent", "Previous_Score", "Final_Score"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

df["Attendance_Percent"] = df["Attendance_Percent"].clip(0, 100)
df["Previous_Score"] = df["Previous_Score"].clip(0, 100)
df["Final_Score"] = df["Final_Score"].clip(0, 100)
df["Study_Hours"] = df["Study_Hours"].clip(lower=0)
df["Score_Change"] = df["Final_Score"] - df["Previous_Score"]

display(df.head())
print("Cleaned shape:", df.shape)
print("Missing values after cleaning:")
display(df.isna().sum().to_frame("missing_count"))


## 4. Pandas and NumPy operations: filtering, sorting, grouping, and aggregation

In [ ]:
# Filter students with final scores of 80 or above
high_scores = df.loc[df["Final_Score"] >= 80, ["Student_ID", "Name", "Final_Score"]]
display(high_scores)

# Sort from highest to lowest final score
display(df.sort_values("Final_Score", ascending=False)[["Student_ID", "Name", "Final_Score"]])

# Group and aggregate by department
department_summary = df.groupby("Department").agg(
    Student_Count=("Student_ID", "count"),
    Average_Final_Score=("Final_Score", "mean"),
    Average_Study_Hours=("Study_Hours", "mean")
).round(2)
display(department_summary)

# NumPy numerical operation
scores = df["Final_Score"].to_numpy()
print("NumPy mean:", round(float(np.mean(scores)), 2))
print("NumPy standard deviation:", round(float(np.std(scores)), 2))


## 5. Basic statistical analysis

In [ ]:
display(df[numeric_cols].describe().round(2))
correlation = df[["Study_Hours", "Attendance_Percent", "Previous_Score", "Final_Score"]].corr()
display(correlation.round(2))
print("Average final score:", round(df["Final_Score"].mean(), 2))
print("Median final score:", round(df["Final_Score"].median(), 2))
print("Correlation is an association measure and does not prove causation.")


## 6. Data visualization

In [ ]:
# Bar chart: average score by department
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="Department", y="Final_Score", errorbar=None)
plt.title("Average Final Score by Department")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bar_average_score_by_department.png", dpi=150)
plt.show()


In [ ]:
# Line chart: score by student ID
ordered = df.sort_values("Student_ID")
plt.figure(figsize=(8, 5))
plt.plot(ordered["Student_ID"].astype(str), ordered["Final_Score"], marker="o")
plt.title("Final Score by Student ID")
plt.xlabel("Student ID")
plt.ylabel("Final Score")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "line_student_scores.png", dpi=150)
plt.show()


In [ ]:
# Histogram: distribution of final scores
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="Final_Score", bins=6, kde=True)
plt.title("Distribution of Final Scores")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hist_final_score.png", dpi=150)
plt.show()


In [ ]:
# Scatter plot: study hours vs final score
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Study_Hours", y="Final_Score", hue="Department", s=90)
plt.title("Study Hours vs Final Score")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "scatter_study_vs_score.png", dpi=150)
plt.show()


In [ ]:
# Box plot: score spread by department
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Department", y="Final_Score")
plt.title("Final Score Spread by Department")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "box_final_score_department.png", dpi=150)
plt.show()


## 7. Save cleaned data and document observations

### Observations
1. Department averages can be compared using the grouped summary and bar chart.
2. The scatter plot helps inspect whether study hours and final scores move together.
3. The histogram shows how final scores are distributed.
4. Median imputation keeps rows with missing numeric values for this small practice dataset.
5. Duplicate records and duplicate student IDs are removed. The sample dataset is fictional.

**Conclusion:** The notebook demonstrates a complete introductory data-science workflow: load, inspect, clean, analyze, visualize, and save.

In [ ]:
cleaned_path = OUTPUT_DIR / "cleaned_student_performance.csv"
df.to_csv(cleaned_path, index=False)
print("Cleaned dataset saved to:", cleaned_path)
print("Charts saved to:", OUTPUT_DIR)
print("Task completed successfully.")
